[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ananttripathi/hybrid-rag-customer-support/blob/main/RAG_Implementation.ipynb)


# **Notebook 4: RAG Implementation**
## Assignment: Hybrid RAG & Fine-Tuning for Customer Support
---

### TO-DO: Before Running This Notebook

**Files you NEED:**
- [ ] `corporate_policies/` folder with `.md` SOP files
- [ ] `outputs.json` — Created by Notebook 3
- [ ] GPU runtime enabled

**Files this notebook will CREATE:**
- [ ] `./chroma_db/` — Persisted ChromaDB vector index _(Required by NB5 and NB7)_
- [ ] `outputs.json` (updated) — adds `naive_rag_output` _(Required by NB5 and NB7)_

---

### **Task 3.2: Implement Retrieval-Assisted Generation**

#### **3.2.1 Generate Embeddings [4 marks]**
**The Task:** Initialise the `all-MiniLM-L6-v2` embedding model, embed the SOP documents, and validate the embeddings were produced.

**Hints & Tips:**
* Load the SOP documents with `TextLoader` first (or reuse the corpus from NB2).
* `HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")` runs on CPU — no GPU needed.
* Validate by embedding one sample string and checking the vector length (384 dims for MiniLM).
* You MUST use the same embedding model when reloading in Notebooks 5 and 7.

**Embedding Model Options:**
* **`all-MiniLM-L6-v2`** (recommended): 384-dim, fast, ~80MB.
* **`all-mpnet-base-v2`**: 768-dim, higher quality, slower.
* **`bge-small-en-v1.5`**: 384-dim, newer architecture.

**Learner Inference:** Your text is now coordinates in semantic space — similar meanings sit close together.

In [1]:
import json
import pickle
from langchain_huggingface import HuggingFaceEmbeddings

with open("retrieval_docs.pkl", "rb") as f:
    retrieval_docs = pickle.load(f)
print(f"Loaded {len(retrieval_docs)} retrieval documents from Notebook 2.")

# Same embedding model MUST be reused when reloading ChromaDB in Notebooks 5 and 7.
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
embeddings = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL)

sample_vec = embeddings.embed_query(retrieval_docs[0].page_content)
print(f"\nEmbedding model: {EMBEDDING_MODEL}")
print(f"Sample embedding dimension: {len(sample_vec)}")
assert len(sample_vec) == 384, "Expected 384-dim embeddings from all-MiniLM-L6-v2."

doc_vectors = embeddings.embed_documents([d.page_content for d in retrieval_docs])
print(f"Embedded {len(doc_vectors)} documents, each of dimension {len(doc_vectors[0])}.")
print("Embedding validation passed.")


Loaded 13 retrieval documents from Notebook 2.


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]


Embedding model: sentence-transformers/all-MiniLM-L6-v2
Sample embedding dimension: 384
Embedded 13 documents, each of dimension 384.
Embedding validation passed.


#### **3.2.2 Build Vector Index [4 marks]**
**The Task:** Create a persistent Chroma vector index from the embedded documents, configure similarity search, and validate the index.

**Hints & Tips:**
* `Chroma.from_documents(docs, embeddings, persist_directory="./chroma_db")` auto-saves — no manual `.persist()` needed.
* Validate with `vector_db._collection.count()` — should equal the number of SOP documents.
* Run one test `.similarity_search("refund", k=1)` to confirm retrieval works.

**Vector DB Options:**
* **ChromaDB** (recommended): simple API, auto-persistence, LangChain integration.
* **FAISS**: faster for >100K docs, but no built-in persistence (manual serialization).

**Learner Inference:** The index lets you search by meaning — it returns the document mathematically closest to your query's coordinates.

In [2]:
import os
import shutil
from langchain_chroma import Chroma

CHROMA_DIR = "./chroma_db"
if os.path.exists(CHROMA_DIR):
    shutil.rmtree(CHROMA_DIR)  # rebuild fresh so the index always matches the current corpus

vector_db = Chroma.from_documents(retrieval_docs, embeddings, persist_directory=CHROMA_DIR)

count = vector_db._collection.count()
print(f"ChromaDB index built and persisted to '{CHROMA_DIR}'.")
print(f"Document count in index: {count} (expected {len(retrieval_docs)})")
assert count == len(retrieval_docs), "Chroma index document count mismatch."

test_results = vector_db.similarity_search("refund", k=1)
print("\nValidation query 'refund' retrieved:", test_results[0].metadata["source_file"])
print(test_results[0].page_content[:150], "...")
print("\nIndex validated: similarity search returns results.")


ChromaDB index built and persisted to './chroma_db'.
Document count in index: 13 (expected 13)



Validation query 'refund' retrieved: refund_policy.md
# Refund Policy

## Eligibility
Customers may request a refund within 30 days of the original purchase or
delivery date, whichever is later. To be eli ...

Index validated: similarity search returns results.


#### **3.2.3 Implement Retrieval Workflow [4 marks]**
**The Task:** Execute a "Naive RAG" workflow — pass the raw customer query into the vector DB, fetch the top result, and augment the LLM prompt.

**Hints & Tips:**
* Use `.similarity_search(query, k=1)` for the top-1 document.
* Check whether the raw query retrieved the WRONG policy — common with ambiguous queries.
* Inject context via the system prompt: `"Answer strictly using this SOP: {context}"`.

**Parameter Tuning:**
* `k=1`: one document (focused). `k=3`: more context if SOPs overlap. `k=5`: max, risks long prompts.

**Learner Inference:** Noisy queries often retrieve the wrong document — proving Naive RAG is flawed and motivating the fine-tuned router.

In [3]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

with open("outputs.json") as f:
    outputs = json.load(f)
test_query = outputs["test_query"]

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"  # same MODEL_ID as every other notebook
DEVICE = "mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
base_model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.float16).to(DEVICE)
base_model.eval()

GEN_KWARGS = dict(max_new_tokens=120, do_sample=False, temperature=None, top_p=None)

# --- Naive retrieval: embed the RAW customer query, fetch top-1 SOP, inject as context ---
retrieved = vector_db.similarity_search(test_query, k=1)
retrieved_doc = retrieved[0]
print("Raw (noisy) query:", test_query)
print("Retrieved SOP:", retrieved_doc.metadata["source_file"])
print(retrieved_doc.page_content[:300], "...")
is_expected = retrieved_doc.metadata["source_file"] == "shipping_delays.md"
print(f"\nRetrieved the expected shipping_delays.md? {is_expected}")
if not is_expected:
    print("Naive retrieval picked the WRONG document for this noisy/sarcastic query — "
          "exactly the failure mode Hybrid RAG (Stage 4) is designed to fix.")

RAG_SYSTEM_TEMPLATE = (
    "You are a customer support assistant. Answer strictly using this SOP:\n\n{context}\n\n"
    "If the SOP does not cover the question, say you will escalate to a human agent."
)

def generate_naive_rag(query, k=1):
    docs = vector_db.similarity_search(query, k=k)
    context = "\n\n".join(d.page_content for d in docs)
    system_prompt = RAG_SYSTEM_TEMPLATE.format(context=context)
    messages = [{"role": "system", "content": system_prompt}, {"role": "user", "content": query}]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        out = base_model.generate(**inputs, pad_token_id=tokenizer.pad_token_id, **GEN_KWARGS)
    answer = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
    return answer, docs

naive_rag_output, retrieved_docs_for_query = generate_naive_rag(test_query, k=1)
print("\nNAIVE RAG OUTPUT (context-augmented):\n", naive_rag_output)


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Raw (noisy) query: my package is still not here and its been forever, this is ridiculous, where even is it??
Retrieved SOP: shipping_delays.md
# Shipping Delays

## Scope
This procedure covers orders that have shipped but are running behind the
estimated delivery date, as well as orders stuck in a pre-shipment state.

## Standard Timelines
Domestic orders deliver within 3–7 business days; international orders within
10–21 business days. An ...

Retrieved the expected shipping_delays.md? True



NAIVE RAG OUTPUT (context-augmented):
 I'm sorry to hear that your package hasn't arrived yet. I understand how frustrating this must be. To help us better assist you, could you please provide me with the following information:
1. The order number (if available).
2. Your contact details (name, email, phone number) so we can reach out directly if needed.
3. Any other relevant details about the item being shipped (e.g., size, weight).

Once I have this information, I'll work with our team to ensure your package reaches you as soon as possible. Thank you for bringing this issue to our attention


---
## Save Artifacts for Downstream Notebooks

In [4]:
outputs["naive_rag_output"] = naive_rag_output
outputs["naive_rag_retrieved_doc"] = retrieved_docs_for_query[0].metadata["source_file"]

with open("outputs.json", "w") as f:
    json.dump(outputs, f, indent=2)

print("Updated outputs.json:")
print(json.dumps(outputs, indent=2))
print(f"\n'{CHROMA_DIR}' persisted on disk for reuse in Notebooks 5 and 7.")


Updated outputs.json:
{
  "test_query": "my package is still not here and its been forever, this is ridiculous, where even is it??",
  "ground_truth": "Domestic orders deliver within 3-7 business days. If a shipment exceeds this window, escalate per the Shipping Delays SOP.",
  "baseline_output": "I'm sorry to hear that your package has not arrived yet. It's understandable if you're feeling frustrated. Here are some steps you can take:\n\n1. Check the tracking information: Make sure you have access to the tracking number for your package. You should be able to find this on the website or app where you placed your order.\n\n2. Contact the shipping company: If you haven't already done so, reach out to the shipping company responsible for delivering your package. They may be able to provide more information about why your package hasn't arrived yet.\n\n3. Check your address: Double-check",
  "naive_rag_output": "I'm sorry to hear that your package hasn't arrived yet. I understand how frus

---
## END-OF-NOTEBOOK CHECKLIST

> **IMPORTANT: Verify before proceeding to Notebook 5.**

- [ ] SOP documents loaded via `TextLoader`
- [ ] **Embeddings generated and validated** ← _Task 3.2.1_
- [ ] **ChromaDB index built, validated, and persisted** ← _Task 3.2.2_
- [ ] Naive similarity search executed on `test_query`
- [ ] Naive RAG output generated with SOP context injected
- [ ] **`./chroma_db/` exists on disk** ← _CRITICAL for NB5 and NB7_
- [ ] **`outputs.json` updated** with `naive_rag_output` ← _CRITICAL for NB5 and NB7_

**If any item is unchecked, fix it before moving on.**